# 3.2 Evaluate clustering performance
In this exercise you will evaluate the clustering performance of your K-means implementation. To do so, you will exploit the Silhouette measure.

> 1. Design and implement two different functions to compute the Silhouette. One should compute the metric for each sample, the other should compute the average silhouette score (you will find several evaluation scores like this in scikit-learn). Try to solve this exercise by using numpy APIs.You can start from this structure:

In [ ]:
def silhouette_samples (X, labels):
    """
    Evaluate the silhouette for each point and return them as a list.
    : param X: input data points, array, shape = (N,C).
    :param labels: the list of cluster labels, shape = N.
    :return: silhouette: array, shape = N
    """
    pass

def silhouette_score (X, labels):
    """
    Evaluate the silhouette for each point and return the mean.
    : param X: input data points, array,
    shape = (N, C).
    :param labels: the list of cluster labels, shape = N.
    : return: silhouette: float
    """
    pass

# How does the silhouette work?
#### **STEP 1**
For each point, look inside its cluster → a(i)

Pick a point i.  
	1.	Look at all the other points in the same cluster as i.  
	2.	Compute the distance from i to each of them.  
	3.	Take the average of those distances.  
So for each point inside a cluster compute the average distance from all points of the cluster to that point. Compute this average distance for all points inside the cluster.

#### **STEP 2**
For the **same point**, look at other clusters → b(i)  

Now ignore the cluster of i.  

For each other cluster C:  
	1.	Compute the distance from i to all points in C.  
	2.	Take the average distance to points in C.  

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# from sklearn.cluster import KMeans

In [ ]:
class KMeans_HM:
    def __init__(self, n_clusters, max_iter=100):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.centroids = None
        self.labels = None
    

    def fit_predict (self, X, gradual_plot = True):
        """Run the K-means clustering on X.
        : param X: input data points, array, shape = (N,C). (5000, 2)
        : return: labels : array, shape = N.
        """

        # ignore, just for the gradual plot
        if gradual_plot:
            plt.figure(figsize=(15,15))
            plot_idx = 1

        # 1. initialize K random centroids in the data space (only once)
        initial_centroids = self.initilization_centroids(X)
    
        for idx in range(self.max_iter):
            # 2. compute distance matrix
            distance_matrix = self.d_from_to_matrix(X, initial_centroids)

            # 3. use the 2D from-to matrix to get closest centroid for each point
            labels = self.get_clusters(distance_matrix)

            # 4. recompute centroids
            centroids = self.get_new_centroids(X, distance_matrix, labels)

            # 5. update the new centroids
            initial_centroids = centroids

            # ignore, just for the gradual plot
            if gradual_plot and idx%10==0:
                plt.subplot(5,3, plot_idx)
                sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
                sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
                plt.title(f"Iteration {idx}")

                plot_idx += 1
        
        # ignore, just for the gradual plot
        if gradual_plot:
            plt.tight_layout()
            plt.show()

        self.labels = labels
        self.centroids = centroids
        return labels, centroids
    






    def initilization_centroids(self, X):
        # get random rows from data as initial centroids
        initial_centroids_idx = np.random.randint(low = 0, high=len(X), size=self.n_clusters)       # take these rows
        initial_centroids = X[initial_centroids_idx]

        return initial_centroids


    def d_from_to_matrix(self, X, initial_centroids):
        # expand dimension to allow broadcasting:
            # build from-to matrix, distance from a point (row) to a centroid (column)
            # (5000, 2)
            # (15, 2)
            # I want (5000, 15), then:
            # (5000, 2) --> (5000,  *1*,  2)
            # (15, 2)   --> (*1*,     15, 2)
        X_expanded = X.reshape(X.shape[0], 1, 2)
        initial_centroids_expanded = initial_centroids.reshape(1, initial_centroids.shape[0], 2)

        # compute euclidean distance + save it in a 2D from-to matrix
        difference = X_expanded - initial_centroids_expanded
        sq_difference = difference**2
        sum_sq_difference = np.sum(sq_difference, axis=2)
        distance_matrix = sum_sq_difference**0.5
        
        return distance_matrix
    
    def get_clusters(self, distance_matrix):
        labels = []
        for row in distance_matrix:
            # pick smallest values from row, USE ARGMIN TO PICK THE INDEX SO THAT YOU ALSO KNOW THE CLUSTER
            row_cluster = row.argmin()
            labels.append(row_cluster)

        return labels

    def get_new_centroids(self, X, distance_matrix, cluster_labels):
        # the new centroid is the **mean** **of the coordinates** of all points inside that cluster
        
        # to make things easier use a df to associate each points to its cluster
        distance_matrix_df = pd.DataFrame(distance_matrix)
        distance_matrix_df['cluster'] = cluster_labels

        # exploit the groupby to group the points that belongs to the same cluster together and use their index to get their coordinates
        centroids = []
        grouped = distance_matrix_df.groupby('cluster')
        for id, values in grouped:
            idx_points_same_cluster = values.index
            # get the coordinates of points with same values, take the X values and do thre mean, same on the Y values = new_centroid
            centroid = [X[idx_points_same_cluster][:,0].mean(), X[idx_points_same_cluster][:,1].mean()]
            centroids.append(centroid)
        
        # convert the list into an array to make things quicker
        centroids = np.array(centroids)
        
        return centroids



# '''--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------'''

def final_plot(X, labels, centroids):
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
    sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
    plt.title('Final clustering')
    plt.show()



def silhouette_samples (X, labels):
    """
    Evaluate the silhouette for each point and return them as a list.
    : param X: input data points, array, shape = (N,C).
    :param labels: the list of cluster labels, shape = N.
    :return: silhouette: array, shape = N
    """
    pass



def silhouette_score (X, labels):
    """
    Evaluate the silhouette for each point and return the mean.
    : param X: input data points, array,
    shape = (N, C).
    :param labels: the list of cluster labels, shape = N.
    : return: silhouette: float
    """
    pass



if __name__ == '__main__':
    df = pd.read_csv('../../Dataset/LAB8/2D_gauss_clusters.txt', sep = ',')
    X = df.values
    
    kmeans = KMeans_HM(15)
    labels, centroids = kmeans.fit_predict(X, gradual_plot=False)
    final_plot(X,labels, centroids)

    

In [ ]:
    # silhouette_samples
# STEP 1: For each point, look inside its cluster
# build a df to link each point to its label/cluster
# group by cluster
# pick one point inside the cluster (fix it) and compute distance to all points (vary them), sum the distances and divide it by the number of points inside the considered cluster (so for each cluster)
# translated:
# for each cluster --> group by cluster and iterate on the groups OR get unique clusters, iterate on them and use masking
    # for each point inside that cluster (fixed point)
        # for each point inside that cluster (varying points)
            # compute distance,
            # sum the distances,
        # divide the sum over the number of points inside the cluster
        # append this score to a list [a(1), a(2), ... a(i)]

X_df = pd.DataFrame(X)
X_df['cluster'] = labels

grouped = X_df.groupby('cluster')
avg_distances_inside_cluster = []
for id, value in grouped:                               # iterate over clusters, WE STAY INSIDE CLUSTERS
    value = value.drop(columns='cluster')
    for point_fix in value.values:                      # fixed point INSIDE CLUSTER
        distances = []
        for point_vary in value.values:                 # varying points INSIDE CLUSTER
            
            # make sure not to compute distance from the same points --> is it really necessary? the distance will be 0, it will give no contribute in the sum, m eh whatever
            
            # compute distance
            difference = point_vary-point_fix
            sq_difference = difference**2
            sum_sq_difference = np.sum(sq_difference)
            distance = sum_sq_difference**0.5
            distances.append(distance)
        
        avg_distance = np.mean(np.array(distances))
        avg_distances_inside_cluster.append(avg_distance)

avg_distances_inside_cluster = np.array(avg_distances_inside_cluster)
print(avg_distances_inside_cluster)
print(len(avg_distances_inside_cluster))

# NOTE:
# this is correct, fix one point, compute distance to all other points, compute the mean, so one single value, repeat it for all points,
# so you'll expect a mean distance associated to each point, as many mean distances as points 5000,
# that's the avg distance from that point to all other points that belongs to its same cluster

# BUT, I created a blank list, I need to keep track of who does that avg distance belongs to, to which point? I need to keep track of indexes

# ex [73864.57496325 89706.08241933 92456.24983731 ... 55296.89747036 48100.99775137 68689.28029857]
# 73864.57496325 is the avg distance from the first point to all other points inside its same cluster ...
# YES BUT NO : IT'S NOT THE FIRST POINT, you don't know who does that avg distance belongs to, it's the first point returned by the groupby

# Also this code has 3 nested for, that's bad man ... is there another way?
# I mean I'm fixing a value and subtracting that value to all other points ... exploit the matrix structure then!

#### You have to stop thinking and working manually iterating on rows / columns and start doing computation between matrixes exploiting whole matrixes

In [ ]:
X_df = pd.DataFrame(X, columns=['x1', 'x2'])
X_df['cluster'] = labels
display(X_df)

In [ ]:
X_df = pd.DataFrame(X, columns=['x1', 'x2'])
X_df['cluster'] = labels

grouped = X_df.groupby('cluster')
# avg_distances_inside_cluster = [] --> use a matrix instead, you already know it will be a matrix with one value associated to each point == len(df)
avg_distances_inside_cluster = np.zeros(len(X_df))

for id, value in grouped:
    # clean value to get only coordinates and indexes so that we can build the previous list but where the first value is the avg distance associated to the first point
    # and convert them to arrays
    coordinates = value[['x1', 'x2']].values        # coordinates of the points inside this cluster
    idxs = np.array(value.index)                    # index of these points inside this cluster
    
    # fix a point and compute distance = subtract the whole matrix of coordinates this fixed value, square, sum over rows, square root
    # I need to use range because idxs has big values like the point 4999 is inside the considered cluster, coordinates doesn't have the index 4999, it has like 30 points, which are the points inside this cluster and the point with index 4999 is the 30th point
    for fixed_point_idx in range(len(idxs)):
        diff = coordinates - coordinates[fixed_point_idx]   # (321, 2)
        sq_diff = diff**2                                   # (321, 2)
        sum_sq_diff = np.sum(sq_diff, axis=1)               # axis = 0 ⬇ (col) ; axis = 1 ➡️ (row) --> after the sum it becomes a single value: (321, 1)
        distances = sum_sq_diff**0.5                        # this is a matrix that contains a single value per point = distance from that point to the fixed point --> we need the average distance from all points inside that cluster to the fixed point                   

        # (321, 1), we need the mean = a single value per fixed point, on average the points inside that cluster differ this much from this fixed point --> compute the mean = sum distances / number of points inside that cluster
        # avg_distance = np.sum(distances) / len(coordinates)
        avg_distance = np.mean(distances)

        # nice, I know I have to add it to avg_distances_inside_cluster, BUT be careful about WHERE, the position of what we want to add
        # idxs is a 1D matrix (list) of indexes, these are the original indexes of the points
        # at the first iteration of the for loop coordinates[fixed_point_idx] is the first point of coordinates = coordinates[0], and the first point of coordinates has index idxs[0]

        idx = idxs[fixed_point_idx]
        
        # avg distance from all points inside the same cluster of the point in index avg_distances_inside_cluster is avg_distance
        avg_distances_inside_cluster[idx] = avg_distance



In [ ]:
X_df = pd.DataFrame(X, columns=['x1', 'x2'])
X_df['cluster'] = labels
X_df['avg_dist_to_points_same_cluster'] = avg_distances_inside_cluster

display(X_df)

In [ ]:
display(X_df)

In [ ]:
    # silhouette_samples
# STEP 2: look outside the cluster to which the point belongs to
# consider one cluster, fix it
# consider one point belonging to that cluster, fix it
# consider another cluster (vary), consider all points inside this other cluster --> NOTE: make sure these two clusters are different
# compute the distance between the fixed point in C0 and all points inside the other considered cluster C1
# average the distances
# redo this for all points of all clusters while the point is still fixed
# pick the minimum distance
# repeat for all points inside C0
# repeat for all clusters

# use masking, not groupby

avg_distances_outside_cluster = np.zeros(len(X_df.index))                                           # initialise 1D matrix, it will have as many values as points, each point will have one value: the min among all avg distances from that point to all points in each cluster

# consider one cluster
for fixed_cluster in X_df['cluster'].unique():                                                      # fix C0
    
    # fix one point inside the considered cluster
    coordinates_inside_cluster = X_df[X_df['cluster'] == fixed_cluster][['x1', 'x2']].values
    idxs = np.array(X_df[X_df['cluster'] == fixed_cluster].index)                                   # real indexes of points : ESSENTIAL TO KEEP TRACK OF THE ORIGINAL POINT AFTER COMPUTING THE AVG DISTANCE

    for idx_fixed_point_inside_cluster in range(len(coordinates_inside_cluster)):
        fixed_point_inside_cluster = coordinates_inside_cluster[idx_fixed_point_inside_cluster]     # fix P1, a point inside C0
        
        avg_distances_outside_cluster_fixed_point = np.zeros(len(X_df['cluster'].unique())-1)         # here I'll expect to have as many values as clusters, one avg distance from fixed point inide C0 to all points in C1, then C2, ecc. one avg distance per cluster

        # vary the other clusters
        i = 0
        for varying_cluster in X_df['cluster'].unique():                                            # pick another cluster C1 (make sure they are NOT the same cluster)
            if varying_cluster == fixed_cluster:
                continue
            else:
                coordinates_outside_cluster = X_df[X_df['cluster'] == varying_cluster][['x1', 'x2']].values
                diff = coordinates_outside_cluster - fixed_point_inside_cluster
                sq_diff = diff**2
                sum_sq_diff = np.sum(sq_diff, axis = 1)
                distance = sum_sq_diff**0.5

                avg_distance_outside_cluster_fixed_point = np.mean(distance)                        # this is a single value, the avg distance from a fixed point inside C0 to all points inside C1
                
                avg_distances_outside_cluster_fixed_point[i] = avg_distance_outside_cluster_fixed_point           # here I store the avg distance from fixed point in C0 to all points in C1, then C2, ecc. and then ...
                
                i += 1
        
        # ... after computing the avg distance for all points in all clusters, take the min
        avg_distance = np.min(avg_distances_outside_cluster_fixed_point)
        
        idx = idxs[idx_fixed_point_inside_cluster]
        avg_distances_outside_cluster[idx] = avg_distance

In [ ]:
X_df = pd.DataFrame(X, columns=['x1', 'x2'])
X_df['cluster'] = labels
X_df['avg_dist_to_points_same_cluster'] = avg_distances_inside_cluster
X_df['avg_dist_to_points_outside_cluster'] = avg_distances_outside_cluster

display(X_df)

#### Jesus christ it was hard as fuck
Now for each point we have the average distance to all points inside its own cluster and the minimum among all the average distances of that points to all other points belonging to a cluster outside the fixed point cluster.  

NOW, what do we do?  
The Silhouette follows this little silly rule:  
For each point $i$:

- $a(i)$: average distance from $i$ to all other points in its own cluster.
- $b(i)$: minimum, over all other clusters $C_k$, of the average distance from $i$ to all points in $C_k$.

The silhouette score for point $i$ is:

$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}$

Overall silhouette for the clustering:

$S = \frac{1}{N} \sum_{i=1}^{N} s(i)$

In [ ]:
# compute silhouette per sample = per each point

a1 = X_df['avg_dist_to_points_same_cluster'].values
b1 = X_df['avg_dist_to_points_outside_cluster'].values
s = np.column_stack((a1, b1))
silhouette_samples = (b1 - a1) / np.max(s, axis=1)
X_df['silhouette_per_point'] = silhouette_samples

display(X_df.sort_values(by='silhouette_per_point', ascending=False))

In [ ]:
# compute silhouette per cluster

silhouette_per_cluster = []
grouped = X_df.groupby('cluster')
for id, values in grouped:
    silhouette_per_cluster.append(np.sum(values['silhouette_per_point']) / len(values))

print(silhouette_per_cluster)
print(len(silhouette_per_cluster))

In [ ]:
# compute a single score
silhouette_score = np.mean(X_df['silhouette_per_point'])
print(silhouette_score)

In [ ]:
class KMeans_HM:
    def __init__(self, n_clusters, max_iter=100):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.centroids = None
        self.labels = None
    

    def fit_predict (self, X, gradual_plot = True):
        """Run the K-means clustering on X.
        : param X: input data points, array, shape = (N,C). (5000, 2)
        : return: labels : array, shape = N.
        """

        # ignore, just for the gradual plot
        if gradual_plot:
            plt.figure(figsize=(15,15))
            plot_idx = 1

        # 1. initialize K random centroids in the data space (only once)
        initial_centroids = self.initilization_centroids(X)
    
        for idx in range(self.max_iter):
            # 2. compute distance matrix
            distance_matrix = self.d_from_to_matrix(X, initial_centroids)

            # 3. use the 2D from-to matrix to get closest centroid for each point
            labels = self.get_clusters(distance_matrix)

            # 4. recompute centroids
            centroids = self.get_new_centroids(X, distance_matrix, labels)

            # 5. update the new centroids
            initial_centroids = centroids

            # ignore, just for the gradual plot
            if gradual_plot and idx%10==0:
                plt.subplot(5,3, plot_idx)
                sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
                sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
                plt.title(f"Iteration {idx}")

                plot_idx += 1
        
        # ignore, just for the gradual plot
        if gradual_plot:
            plt.tight_layout()
            plt.show()

        self.labels = labels
        self.centroids = centroids
        return labels, centroids
    






    def initilization_centroids(self, X):
        # get random rows from data as initial centroids
        initial_centroids_idx = np.random.randint(low = 0, high=len(X), size=self.n_clusters)       # take these rows
        initial_centroids = X[initial_centroids_idx]

        return initial_centroids


    def d_from_to_matrix(self, X, initial_centroids):
        # expand dimension to allow broadcasting:
            # build from-to matrix, distance from a point (row) to a centroid (column)
            # (5000, 2)
            # (15, 2)
            # I want (5000, 15), then:
            # (5000, 2) --> (5000,  *1*,  2)
            # (15, 2)   --> (*1*,     15, 2)
        X_expanded = X.reshape(X.shape[0], 1, 2)
        initial_centroids_expanded = initial_centroids.reshape(1, initial_centroids.shape[0], 2)

        # compute euclidean distance + save it in a 2D from-to matrix
        difference = X_expanded - initial_centroids_expanded
        sq_difference = difference**2
        sum_sq_difference = np.sum(sq_difference, axis=2)
        distance_matrix = sum_sq_difference**0.5
        
        return distance_matrix
    
    def get_clusters(self, distance_matrix):
        labels = []
        for row in distance_matrix:
            # pick smallest values from row, USE ARGMIN TO PICK THE INDEX SO THAT YOU ALSO KNOW THE CLUSTER
            row_cluster = row.argmin()
            labels.append(row_cluster)

        return labels

    def get_new_centroids(self, X, distance_matrix, cluster_labels):
        # the new centroid is the **mean** **of the coordinates** of all points inside that cluster
        
        # to make things easier use a df to associate each points to its cluster
        distance_matrix_df = pd.DataFrame(distance_matrix)
        distance_matrix_df['cluster'] = cluster_labels

        # exploit the groupby to group the points that belongs to the same cluster together and use their index to get their coordinates
        centroids = []
        grouped = distance_matrix_df.groupby('cluster')
        for id, values in grouped:
            idx_points_same_cluster = values.index
            # get the coordinates of points with same values, take the X values and do thre mean, same on the Y values = new_centroid
            centroid = [X[idx_points_same_cluster][:,0].mean(), X[idx_points_same_cluster][:,1].mean()]
            centroids.append(centroid)
        
        # convert the list into an array to make things quicker
        centroids = np.array(centroids)
        
        return centroids



# '''--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------'''

def final_plot(X, labels, centroids):
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
    sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
    plt.title('Final clustering')
    plt.show()




def get_X_df(X, labels):
    X_df = pd.DataFrame(X, columns=['x1', 'x2'])
    X_df['cluster'] = labels
    return X_df


def get_avg_distances_inside_cluster(X_df):
    grouped = X_df.groupby('cluster')
    avg_distances_inside_cluster = np.zeros(len(X_df))

    for id, value in grouped:
        coordinates = value[['x1', 'x2']].values        # coordinates of the points inside this cluster
        idxs = np.array(value.index)                    # index of these points inside this cluster
        
        # fix a point and compute distance = subtract the whole matrix of coordinates this fixed value, square, sum over rows, square root
        # I need to use range because idxs has big values like the point 4999 is inside the considered cluster, coordinates doesn't have the index 4999, it has like 30 points, which are the points inside this cluster and the point with index 4999 is the 30th point
        for fixed_point_idx in range(len(idxs)):
            diff = coordinates - coordinates[fixed_point_idx]   # (321, 2)
            sq_diff = diff**2                                   # (321, 2)
            sum_sq_diff = np.sum(sq_diff, axis=1)               # axis = 0 ⬇ (col) ; axis = 1 ➡️ (row) --> after the sum it becomes a single value: (321, 1)
            distances = sum_sq_diff**0.5                        # this is a matrix that contains a single value per point = distance from that point to the fixed point --> we need the average distance from all points inside that cluster to the fixed point                   

            avg_distance = np.mean(distances)

            idx = idxs[fixed_point_idx]
            
            avg_distances_inside_cluster[idx] = avg_distance
    
    return avg_distances_inside_cluster


def get_avg_distances_outside_cluster(X_df):
    avg_distances_outside_cluster = np.zeros(len(X_df.index))                                           # initialise 1D matrix, it will have as many values as points, each point will have one value: the min among all avg distances from that point to all points in each cluster

    # consider one cluster
    for fixed_cluster in X_df['cluster'].unique():                                                      # fix C0
        
        # fix one point inside the considered cluster
        coordinates_inside_cluster = X_df[X_df['cluster'] == fixed_cluster][['x1', 'x2']].values
        idxs = np.array(X_df[X_df['cluster'] == fixed_cluster].index)                                   # real indexes of points : ESSENTIAL TO KEEP TRACK OF THE ORIGINAL POINT AFTER COMPUTING THE AVG DISTANCE

        for idx_fixed_point_inside_cluster in range(len(coordinates_inside_cluster)):
            fixed_point_inside_cluster = coordinates_inside_cluster[idx_fixed_point_inside_cluster]     # fix P1, a point inside C0
            
            avg_distances_outside_cluster_fixed_point = np.zeros(len(X_df['cluster'].unique())-1)         # here I'll expect to have as many values as clusters, one avg distance from fixed point inide C0 to all points in C1, then C2, ecc. one avg distance per cluster

            # vary the other clusters
            i = 0
            for varying_cluster in X_df['cluster'].unique():                                            # pick another cluster C1 (make sure they are NOT the same cluster)
                if varying_cluster == fixed_cluster:
                    continue
                else:
                    coordinates_outside_cluster = X_df[X_df['cluster'] == varying_cluster][['x1', 'x2']].values
                    diff = coordinates_outside_cluster - fixed_point_inside_cluster
                    sq_diff = diff**2
                    sum_sq_diff = np.sum(sq_diff, axis = 1)
                    distance = sum_sq_diff**0.5

                    avg_distance_outside_cluster_fixed_point = np.mean(distance)                        # this is a single value, the avg distance from a fixed point inside C0 to all points inside C1
                    
                    avg_distances_outside_cluster_fixed_point[i] = avg_distance_outside_cluster_fixed_point           # here I store the avg distance from fixed point in C0 to all points in C1, then C2, ecc. and then ...
                    
                    i += 1
            
            # ... after computing the avg distance for all points in all clusters, take the min
            avg_distance = np.min(avg_distances_outside_cluster_fixed_point)
            
            idx = idxs[idx_fixed_point_inside_cluster]
            avg_distances_outside_cluster[idx] = avg_distance
    
    return avg_distances_outside_cluster


def update_df(X_df, avg_distances_inside_cluster, avg_distances_outside_cluster):
    X_df['avg_dist_to_points_same_cluster'] = avg_distances_inside_cluster
    X_df['avg_dist_to_points_outside_cluster'] = avg_distances_outside_cluster
    
    return X_df


def silhouette_samples(X, labels):
    """
    Evaluate the silhouette for each point and return them as a list.
    : param X: input data points, array, shape = (N,C).
    :param labels: the list of cluster labels, shape = N.
    :return: silhouette: array, shape = N
    """
    # use a df to keep things clean
    X_df = get_X_df(X, labels)

    # get a(i): average distance from each point inside a cluster to all points inside the same cluster, repeat for each cluster and each point
    avg_distances_inside_cluster = get_avg_distances_inside_cluster(X_df)
    
    # get b(i): the minimum average distance between a fixed point inside a cluster and for each other cluster all points inside that cluster
    avg_distances_outside_cluster = get_avg_distances_outside_cluster(X_df)

    # add new columns to df
    X_df = update_df(X_df, avg_distances_inside_cluster, avg_distances_outside_cluster)

    # compute silhouette_samples = silhouette per point
    a1 = X_df['avg_dist_to_points_same_cluster'].values
    b1 = X_df['avg_dist_to_points_outside_cluster'].values
    s = np.column_stack((a1, b1))
    silhouette_samples = (b1 - a1) / np.max(s, axis=1)
    X_df['silhouette_per_point'] = silhouette_samples
    
    return X_df['silhouette_per_point'].values


def silhouette_score (X, labels):
    """
    Evaluate the silhouette for each point and return the mean.
    : param X: input data points, array,
    shape = (N, C).
    :param labels: the list of cluster labels, shape = N.
    : return: silhouette: float
    """

    # use a df to keep things clean
    X_df = get_X_df(X, labels)

    # get a(i): average distance from each point inside a cluster to all points inside the same cluster, repeat for each cluster and each point
    avg_distances_inside_cluster = get_avg_distances_inside_cluster(X_df)
    
    # get b(i): the minimum average distance between a fixed point inside a cluster and for each other cluster all points inside that cluster
    avg_distances_outside_cluster = get_avg_distances_outside_cluster(X_df)

    # add new columns to df
    X_df = update_df(X_df, avg_distances_inside_cluster, avg_distances_outside_cluster)

    # compute silhouette_samples = silhouette per point
    a1 = X_df['avg_dist_to_points_same_cluster'].values
    b1 = X_df['avg_dist_to_points_outside_cluster'].values
    s = np.column_stack((a1, b1))
    silhouette_samples = (b1 - a1) / np.max(s, axis=1)
    X_df['silhouette_per_point'] = silhouette_samples

    # compute overall silhouette
    silhouette_score = np.mean(X_df['silhouette_per_point'])

    return silhouette_score
    
    

if __name__ == '__main__':
    df = pd.read_csv('../../Dataset/LAB8/2D_gauss_clusters.txt', sep = ',')
    X = df.values
    
    kmeans = KMeans_HM(15)
    labels, centroids = kmeans.fit_predict(X, gradual_plot=False)
    final_plot(X,labels, centroids)

    silhouette_per_point = silhouette_samples(X,labels)
    silhouette_overall = silhouette_score(X,labels)
    print(silhouette_overall)

In [ ]:
class KMeans_HM:
    def __init__(self, n_clusters, max_iter=100):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.centroids = None
        self.labels = None
    

    def fit_predict (self, X, gradual_plot = True):
        """Run the K-means clustering on X.
        : param X: input data points, array, shape = (N,C). (5000, 2)
        : return: labels : array, shape = N.
        """

        # ignore, just for the gradual plot
        if gradual_plot:
            plt.figure(figsize=(15,15))
            plot_idx = 1

        # 1. initialize K random centroids in the data space (only once)
        initial_centroids = self.initilization_centroids(X)
    
        for idx in range(self.max_iter):
            # 2. compute distance matrix
            distance_matrix = self.d_from_to_matrix(X, initial_centroids)

            # 3. use the 2D from-to matrix to get closest centroid for each point
            labels = self.get_clusters(distance_matrix)

            # 4. recompute centroids
            centroids = self.get_new_centroids(X, distance_matrix, labels)

            # 5. update the new centroids
            initial_centroids = centroids

            # ignore, just for the gradual plot
            if gradual_plot and idx%10==0:
                plt.subplot(5,3, plot_idx)
                sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
                sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
                plt.title(f"Iteration {idx}")

                plot_idx += 1
        
        # ignore, just for the gradual plot
        if gradual_plot:
            plt.tight_layout()
            plt.show()

        self.labels = labels
        self.centroids = centroids
        return labels, centroids
    






    def initilization_centroids(self, X):
        # get random rows from data as initial centroids
        initial_centroids_idx = np.random.randint(low = 0, high=len(X), size=self.n_clusters)       # take these rows
        initial_centroids = X[initial_centroids_idx]

        return initial_centroids


    def d_from_to_matrix(self, X, initial_centroids):
        # expand dimension to allow broadcasting:
            # build from-to matrix, distance from a point (row) to a centroid (column)
            # (5000, 2)
            # (15, 2)
            # I want (5000, 15), then:
            # (5000, 2) --> (5000,  *1*,  2)
            # (15, 2)   --> (*1*,     15, 2)
        X_expanded = X.reshape(X.shape[0], 1, 2)
        initial_centroids_expanded = initial_centroids.reshape(1, initial_centroids.shape[0], 2)

        # compute euclidean distance + save it in a 2D from-to matrix
        difference = X_expanded - initial_centroids_expanded
        sq_difference = difference**2
        sum_sq_difference = np.sum(sq_difference, axis=2)
        distance_matrix = sum_sq_difference**0.5
        
        return distance_matrix
    
    def get_clusters(self, distance_matrix):
        labels = []
        for row in distance_matrix:
            # pick smallest values from row, USE ARGMIN TO PICK THE INDEX SO THAT YOU ALSO KNOW THE CLUSTER
            row_cluster = row.argmin()
            labels.append(row_cluster)

        return labels

    def get_new_centroids(self, X, distance_matrix, cluster_labels):
        # the new centroid is the **mean** **of the coordinates** of all points inside that cluster
        
        # to make things easier use a df to associate each points to its cluster
        distance_matrix_df = pd.DataFrame(distance_matrix)
        distance_matrix_df['cluster'] = cluster_labels

        # exploit the groupby to group the points that belongs to the same cluster together and use their index to get their coordinates
        centroids = []
        grouped = distance_matrix_df.groupby('cluster')
        for id, values in grouped:
            idx_points_same_cluster = values.index
            # get the coordinates of points with same values, take the X values and do thre mean, same on the Y values = new_centroid
            centroid = [X[idx_points_same_cluster][:,0].mean(), X[idx_points_same_cluster][:,1].mean()]
            centroids.append(centroid)
        
        # convert the list into an array to make things quicker
        centroids = np.array(centroids)
        
        return centroids



# '''--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------'''

def final_plot(X, labels, centroids):
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
    sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
    plt.title('Final clustering')
    plt.show()




def get_X_df(X, labels):
    X_df = pd.DataFrame(X, columns=['x1', 'x2'])
    X_df['cluster'] = labels
    return X_df


def get_avg_distances_inside_cluster(X_df):
    grouped = X_df.groupby('cluster')
    avg_distances_inside_cluster = np.zeros(len(X_df))

    for id, value in grouped:
        coordinates = value[['x1', 'x2']].values        # coordinates of the points inside this cluster
        idxs = np.array(value.index)                    # index of these points inside this cluster
        
        # fix a point and compute distance = subtract the whole matrix of coordinates this fixed value, square, sum over rows, square root
        # I need to use range because idxs has big values like the point 4999 is inside the considered cluster, coordinates doesn't have the index 4999, it has like 30 points, which are the points inside this cluster and the point with index 4999 is the 30th point
        for fixed_point_idx in range(len(idxs)):
            diff = coordinates - coordinates[fixed_point_idx]   # (321, 2)
            sq_diff = diff**2                                   # (321, 2)
            sum_sq_diff = np.sum(sq_diff, axis=1)               # axis = 0 ⬇ (col) ; axis = 1 ➡️ (row) --> after the sum it becomes a single value: (321, 1)
            distances = sum_sq_diff**0.5                        # this is a matrix that contains a single value per point = distance from that point to the fixed point --> we need the average distance from all points inside that cluster to the fixed point                   

            avg_distance = np.mean(distances)

            idx = idxs[fixed_point_idx]
            
            avg_distances_inside_cluster[idx] = avg_distance
    
    return avg_distances_inside_cluster


def get_avg_distances_outside_cluster(X_df):
    avg_distances_outside_cluster = np.zeros(len(X_df.index))                                           # initialise 1D matrix, it will have as many values as points, each point will have one value: the min among all avg distances from that point to all points in each cluster

    # consider one cluster
    for fixed_cluster in X_df['cluster'].unique():                                                      # fix C0
        
        # fix one point inside the considered cluster
        coordinates_inside_cluster = X_df[X_df['cluster'] == fixed_cluster][['x1', 'x2']].values
        idxs = np.array(X_df[X_df['cluster'] == fixed_cluster].index)                                   # real indexes of points : ESSENTIAL TO KEEP TRACK OF THE ORIGINAL POINT AFTER COMPUTING THE AVG DISTANCE

        for idx_fixed_point_inside_cluster in range(len(coordinates_inside_cluster)):
            fixed_point_inside_cluster = coordinates_inside_cluster[idx_fixed_point_inside_cluster]     # fix P1, a point inside C0
            
            avg_distances_outside_cluster_fixed_point = np.zeros(len(X_df['cluster'].unique())-1)         # here I'll expect to have as many values as clusters, one avg distance from fixed point inide C0 to all points in C1, then C2, ecc. one avg distance per cluster

            # vary the other clusters
            i = 0
            for varying_cluster in X_df['cluster'].unique():                                            # pick another cluster C1 (make sure they are NOT the same cluster)
                if varying_cluster == fixed_cluster:
                    continue
                else:
                    coordinates_outside_cluster = X_df[X_df['cluster'] == varying_cluster][['x1', 'x2']].values
                    diff = coordinates_outside_cluster - fixed_point_inside_cluster
                    sq_diff = diff**2
                    sum_sq_diff = np.sum(sq_diff, axis = 1)
                    distance = sum_sq_diff**0.5

                    avg_distance_outside_cluster_fixed_point = np.mean(distance)                        # this is a single value, the avg distance from a fixed point inside C0 to all points inside C1
                    
                    avg_distances_outside_cluster_fixed_point[i] = avg_distance_outside_cluster_fixed_point           # here I store the avg distance from fixed point in C0 to all points in C1, then C2, ecc. and then ...
                    
                    i += 1
            
            # ... after computing the avg distance for all points in all clusters, take the min
            avg_distance = np.min(avg_distances_outside_cluster_fixed_point)
            
            idx = idxs[idx_fixed_point_inside_cluster]
            avg_distances_outside_cluster[idx] = avg_distance
    
    return avg_distances_outside_cluster


def update_df(X_df, avg_distances_inside_cluster, avg_distances_outside_cluster):
    X_df['avg_dist_to_points_same_cluster'] = avg_distances_inside_cluster
    X_df['avg_dist_to_points_outside_cluster'] = avg_distances_outside_cluster
    
    return X_df


def silhouette_samples(X, labels):
    """
    Evaluate the silhouette for each point and return them as a list.
    : param X: input data points, array, shape = (N,C).
    :param labels: the list of cluster labels, shape = N.
    :return: silhouette: array, shape = N
    """
    # use a df to keep things clean
    X_df = get_X_df(X, labels)

    # get a(i): average distance from each point inside a cluster to all points inside the same cluster, repeat for each cluster and each point
    avg_distances_inside_cluster = get_avg_distances_inside_cluster(X_df)
    
    # get b(i): the minimum average distance between a fixed point inside a cluster and for each other cluster all points inside that cluster
    avg_distances_outside_cluster = get_avg_distances_outside_cluster(X_df)

    # add new columns to df
    X_df = update_df(X_df, avg_distances_inside_cluster, avg_distances_outside_cluster)

    # compute silhouette_samples = silhouette per point
    a1 = X_df['avg_dist_to_points_same_cluster'].values
    b1 = X_df['avg_dist_to_points_outside_cluster'].values
    s = np.column_stack((a1, b1))
    silhouette_samples = (b1 - a1) / np.max(s, axis=1)
    X_df['silhouette_per_point'] = silhouette_samples
    
    return X_df['silhouette_per_point'].values


def silhouette_score (X, labels):
    """
    Evaluate the silhouette for each point and return the mean.
    : param X: input data points, array,
    shape = (N, C).
    :param labels: the list of cluster labels, shape = N.
    : return: silhouette: float
    """

    # use a df to keep things clean
    X_df = get_X_df(X, labels)

    # get a(i): average distance from each point inside a cluster to all points inside the same cluster, repeat for each cluster and each point
    avg_distances_inside_cluster = get_avg_distances_inside_cluster(X_df)
    
    # get b(i): the minimum average distance between a fixed point inside a cluster and for each other cluster all points inside that cluster
    avg_distances_outside_cluster = get_avg_distances_outside_cluster(X_df)

    # add new columns to df
    X_df = update_df(X_df, avg_distances_inside_cluster, avg_distances_outside_cluster)

    # compute silhouette_samples = silhouette per point
    a1 = X_df['avg_dist_to_points_same_cluster'].values
    b1 = X_df['avg_dist_to_points_outside_cluster'].values
    s = np.column_stack((a1, b1))
    silhouette_samples = (b1 - a1) / np.max(s, axis=1)
    X_df['silhouette_per_point'] = silhouette_samples

    # compute overall silhouette
    silhouette_score = np.mean(X_df['silhouette_per_point'])

    return silhouette_score
    
    

if __name__ == '__main__':
    df = pd.read_csv('../../Dataset/LAB8/chameleon_clusters.txt', sep = ',')
    X = df.values
    
    kmeans = KMeans_HM(15)
    labels, centroids = kmeans.fit_predict(X, gradual_plot=False)
    final_plot(X,labels, centroids)

    silhouette_per_point = silhouette_samples(X,labels)
    silhouette_overall = silhouette_score(X,labels)
    print(silhouette_overall)

# THIS WAS SO FRICKING HARD MAN!!
Computationally horrible, it takes so much time  
BUT we did it :)

---

#### 2. Implement a function to plot the silhouette values sorted in ascending order.
> **This kind of chart is particularly useful to inspect the overall performance of a clustering technique.**  
In an **ideal case**, the curve is heavily **shifted towards the value 1 on the y-axis**, i.e. most of the points have been assigned coherently.

Create the chart for both your datasets (Synthetic 2-D Gaussian clusters and chamaleon DS) and discuss the results.

In [ ]:
# this is for synthetic 2-D Gaussian clusters
# I just want to plot the SORTED silhouette values per each point, so for x values just take range(1,len(X_df))
x = range(len(X_df))
y = X_df['silhouette_per_point'].sort_values()

plt.figure(figsize=(8,6))
plt.plot(x, y)
plt.title('silhouette across all points')
plt.ylabel('silhouette score of each point')
plt.xlabel('points')
plt.show()

In [ ]:
# Let's do this for the Chamaleon DS too

class KMeans_HM:
    def __init__(self, n_clusters, max_iter=100):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.centroids = None
        self.labels = None
    

    def fit_predict (self, X, gradual_plot = True):
        """Run the K-means clustering on X.
        : param X: input data points, array, shape = (N,C). (5000, 2)
        : return: labels : array, shape = N.
        """

        # ignore, just for the gradual plot
        if gradual_plot:
            plt.figure(figsize=(15,15))
            plot_idx = 1

        # 1. initialize K random centroids in the data space (only once)
        initial_centroids = self.initilization_centroids(X)
    
        for idx in range(self.max_iter):
            # 2. compute distance matrix
            distance_matrix = self.d_from_to_matrix(X, initial_centroids)

            # 3. use the 2D from-to matrix to get closest centroid for each point
            labels = self.get_clusters(distance_matrix)

            # 4. recompute centroids
            centroids = self.get_new_centroids(X, distance_matrix, labels)

            # 5. update the new centroids
            initial_centroids = centroids

            # ignore, just for the gradual plot
            if gradual_plot and idx%10==0:
                plt.subplot(5,3, plot_idx)
                sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
                sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
                plt.title(f"Iteration {idx}")

                plot_idx += 1
        
        # ignore, just for the gradual plot
        if gradual_plot:
            plt.tight_layout()
            plt.show()

        self.labels = labels
        self.centroids = centroids
        return labels, centroids
    






    def initilization_centroids(self, X):
        # get random rows from data as initial centroids
        initial_centroids_idx = np.random.randint(low = 0, high=len(X), size=self.n_clusters)       # take these rows
        initial_centroids = X[initial_centroids_idx]

        return initial_centroids


    def d_from_to_matrix(self, X, initial_centroids):
        # expand dimension to allow broadcasting:
            # build from-to matrix, distance from a point (row) to a centroid (column)
            # (5000, 2)
            # (15, 2)
            # I want (5000, 15), then:
            # (5000, 2) --> (5000,  *1*,  2)
            # (15, 2)   --> (*1*,     15, 2)
        X_expanded = X.reshape(X.shape[0], 1, 2)
        initial_centroids_expanded = initial_centroids.reshape(1, initial_centroids.shape[0], 2)

        # compute euclidean distance + save it in a 2D from-to matrix
        difference = X_expanded - initial_centroids_expanded
        sq_difference = difference**2
        sum_sq_difference = np.sum(sq_difference, axis=2)
        distance_matrix = sum_sq_difference**0.5
        
        return distance_matrix
    
    def get_clusters(self, distance_matrix):
        labels = []
        for row in distance_matrix:
            # pick smallest values from row, USE ARGMIN TO PICK THE INDEX SO THAT YOU ALSO KNOW THE CLUSTER
            row_cluster = row.argmin()
            labels.append(row_cluster)

        return labels

    def get_new_centroids(self, X, distance_matrix, cluster_labels):
        # the new centroid is the **mean** **of the coordinates** of all points inside that cluster
        
        # to make things easier use a df to associate each points to its cluster
        distance_matrix_df = pd.DataFrame(distance_matrix)
        distance_matrix_df['cluster'] = cluster_labels

        # exploit the groupby to group the points that belongs to the same cluster together and use their index to get their coordinates
        centroids = []
        grouped = distance_matrix_df.groupby('cluster')
        for id, values in grouped:
            idx_points_same_cluster = values.index
            # get the coordinates of points with same values, take the X values and do thre mean, same on the Y values = new_centroid
            centroid = [X[idx_points_same_cluster][:,0].mean(), X[idx_points_same_cluster][:,1].mean()]
            centroids.append(centroid)
        
        # convert the list into an array to make things quicker
        centroids = np.array(centroids)
        
        return centroids



# '''--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------'''

def final_plot(X, labels, centroids):
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=False)
    sns.scatterplot(x=centroids[:,0], y=centroids[:,1], s=80, marker='X')
    plt.title('Final clustering')
    plt.show()




def get_X_df(X, labels):
    X_df = pd.DataFrame(X, columns=['x1', 'x2'])
    X_df['cluster'] = labels
    return X_df


def get_avg_distances_inside_cluster(X_df):
    grouped = X_df.groupby('cluster')
    avg_distances_inside_cluster = np.zeros(len(X_df))

    for id, value in grouped:
        coordinates = value[['x1', 'x2']].values        # coordinates of the points inside this cluster
        idxs = np.array(value.index)                    # index of these points inside this cluster
        
        # fix a point and compute distance = subtract the whole matrix of coordinates this fixed value, square, sum over rows, square root
        # I need to use range because idxs has big values like the point 4999 is inside the considered cluster, coordinates doesn't have the index 4999, it has like 30 points, which are the points inside this cluster and the point with index 4999 is the 30th point
        for fixed_point_idx in range(len(idxs)):
            diff = coordinates - coordinates[fixed_point_idx]   # (321, 2)
            sq_diff = diff**2                                   # (321, 2)
            sum_sq_diff = np.sum(sq_diff, axis=1)               # axis = 0 ⬇ (col) ; axis = 1 ➡️ (row) --> after the sum it becomes a single value: (321, 1)
            distances = sum_sq_diff**0.5                        # this is a matrix that contains a single value per point = distance from that point to the fixed point --> we need the average distance from all points inside that cluster to the fixed point                   

            avg_distance = np.mean(distances)

            idx = idxs[fixed_point_idx]
            
            avg_distances_inside_cluster[idx] = avg_distance
    
    return avg_distances_inside_cluster


def get_avg_distances_outside_cluster(X_df):
    avg_distances_outside_cluster = np.zeros(len(X_df.index))                                           # initialise 1D matrix, it will have as many values as points, each point will have one value: the min among all avg distances from that point to all points in each cluster

    # consider one cluster
    for fixed_cluster in X_df['cluster'].unique():                                                      # fix C0
        
        # fix one point inside the considered cluster
        coordinates_inside_cluster = X_df[X_df['cluster'] == fixed_cluster][['x1', 'x2']].values
        idxs = np.array(X_df[X_df['cluster'] == fixed_cluster].index)                                   # real indexes of points : ESSENTIAL TO KEEP TRACK OF THE ORIGINAL POINT AFTER COMPUTING THE AVG DISTANCE

        for idx_fixed_point_inside_cluster in range(len(coordinates_inside_cluster)):
            fixed_point_inside_cluster = coordinates_inside_cluster[idx_fixed_point_inside_cluster]     # fix P1, a point inside C0
            
            avg_distances_outside_cluster_fixed_point = np.zeros(len(X_df['cluster'].unique())-1)         # here I'll expect to have as many values as clusters, one avg distance from fixed point inide C0 to all points in C1, then C2, ecc. one avg distance per cluster

            # vary the other clusters
            i = 0
            for varying_cluster in X_df['cluster'].unique():                                            # pick another cluster C1 (make sure they are NOT the same cluster)
                if varying_cluster == fixed_cluster:
                    continue
                else:
                    coordinates_outside_cluster = X_df[X_df['cluster'] == varying_cluster][['x1', 'x2']].values
                    diff = coordinates_outside_cluster - fixed_point_inside_cluster
                    sq_diff = diff**2
                    sum_sq_diff = np.sum(sq_diff, axis = 1)
                    distance = sum_sq_diff**0.5

                    avg_distance_outside_cluster_fixed_point = np.mean(distance)                        # this is a single value, the avg distance from a fixed point inside C0 to all points inside C1
                    
                    avg_distances_outside_cluster_fixed_point[i] = avg_distance_outside_cluster_fixed_point           # here I store the avg distance from fixed point in C0 to all points in C1, then C2, ecc. and then ...
                    
                    i += 1
            
            # ... after computing the avg distance for all points in all clusters, take the min
            avg_distance = np.min(avg_distances_outside_cluster_fixed_point)
            
            idx = idxs[idx_fixed_point_inside_cluster]
            avg_distances_outside_cluster[idx] = avg_distance
    
    return avg_distances_outside_cluster


def update_df(X_df, avg_distances_inside_cluster, avg_distances_outside_cluster):
    X_df['avg_dist_to_points_same_cluster'] = avg_distances_inside_cluster
    X_df['avg_dist_to_points_outside_cluster'] = avg_distances_outside_cluster
    
    return X_df


def silhouette_samples(X, labels):
    """
    Evaluate the silhouette for each point and return them as a list.
    : param X: input data points, array, shape = (N,C).
    :param labels: the list of cluster labels, shape = N.
    :return: silhouette: array, shape = N
    """
    # use a df to keep things clean
    X_df = get_X_df(X, labels)

    # get a(i): average distance from each point inside a cluster to all points inside the same cluster, repeat for each cluster and each point
    avg_distances_inside_cluster = get_avg_distances_inside_cluster(X_df)
    
    # get b(i): the minimum average distance between a fixed point inside a cluster and for each other cluster all points inside that cluster
    avg_distances_outside_cluster = get_avg_distances_outside_cluster(X_df)

    # add new columns to df
    X_df = update_df(X_df, avg_distances_inside_cluster, avg_distances_outside_cluster)

    # compute silhouette_samples = silhouette per point
    a1 = X_df['avg_dist_to_points_same_cluster'].values
    b1 = X_df['avg_dist_to_points_outside_cluster'].values
    s = np.column_stack((a1, b1))
    silhouette_samples = (b1 - a1) / np.max(s, axis=1)
    X_df['silhouette_per_point'] = silhouette_samples

    plot_silhouette(X_df)
    
    return X_df['silhouette_per_point'].values


def silhouette_score (X, labels):
    """
    Evaluate the silhouette for each point and return the mean.
    : param X: input data points, array,
    shape = (N, C).
    :param labels: the list of cluster labels, shape = N.
    : return: silhouette: float
    """

    # use a df to keep things clean
    X_df = get_X_df(X, labels)

    # get a(i): average distance from each point inside a cluster to all points inside the same cluster, repeat for each cluster and each point
    avg_distances_inside_cluster = get_avg_distances_inside_cluster(X_df)
    
    # get b(i): the minimum average distance between a fixed point inside a cluster and for each other cluster all points inside that cluster
    avg_distances_outside_cluster = get_avg_distances_outside_cluster(X_df)

    # add new columns to df
    X_df = update_df(X_df, avg_distances_inside_cluster, avg_distances_outside_cluster)

    # compute silhouette_samples = silhouette per point
    a1 = X_df['avg_dist_to_points_same_cluster'].values
    b1 = X_df['avg_dist_to_points_outside_cluster'].values
    s = np.column_stack((a1, b1))
    silhouette_samples = (b1 - a1) / np.max(s, axis=1)
    X_df['silhouette_per_point'] = silhouette_samples

    # compute overall silhouette
    silhouette_score = np.mean(X_df['silhouette_per_point'])

    return silhouette_score
    

def plot_silhouette(X_df):
    x = range(len(X_df))
    y = X_df['silhouette_per_point'].sort_values()

    plt.figure(figsize=(8,6))
    plt.plot(x, y)
    plt.title('silhouette across all points')
    plt.ylabel('silhouette score of each point')
    plt.xlabel('points')
    plt.show()

    

if __name__ == '__main__':
    df = pd.read_csv('../../Dataset/LAB8/chameleon_clusters.txt', sep = ',')
    X = df.values
    
    kmeans = KMeans_HM(15)
    labels, centroids = kmeans.fit_predict(X, gradual_plot=False)
    final_plot(X,labels, centroids)


    silhouette_per_point = silhouette_samples(X,labels)
    silhouette_overall = silhouette_score(X,labels)
    print(silhouette_overall)